# 02 · Recolección de datos mediante Web Scraping (Wikipedia)

**Objetivo:** extraer el histórico de lanzamientos Falcon 9 de la página de Wikipedia
*List of Falcon 9 and Falcon Heavy launches* con `requests` + `BeautifulSoup`.

Se usa la revisión fija `oldid=1027686922` (junio 2021), la misma del laboratorio de IBM, para que el
resultado sea reproducible.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
IMAGES_DIR = Path("../images")
for folder in (DATA_DIR, RESULTS_DIR, IMAGES_DIR):
    folder.mkdir(exist_ok=True)

# Repositorio público de datos del curso IBM Applied Data Science Capstone
IBM = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"


def load_csv(name: str, url: str) -> pd.DataFrame:
    """Lee data/<name> si ya existe; si no, lo descarga desde `url` y lo guarda como caché local."""
    local = DATA_DIR / name
    if local.exists():
        print(f"Leyendo {local}")
        return pd.read_csv(local)
    print(f"Descargando {url}")
    df = pd.read_csv(url)
    df.to_csv(local, index=False)
    return df

In [2]:
import json
import re
import unicodedata

import requests
from bs4 import BeautifulSoup

STATIC_URL = (
    "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches"
    "&oldid=1027686922"
)
FALLBACK_CSV_URL = f"{IBM}/datasets/spacex_web_scraped.csv"
HEADERS = {"User-Agent": "ibm-capstone-spacex/1.0 (educational project)"}

## 1. Funciones auxiliares para limpiar las celdas HTML

In [3]:
def date_time(table_cells):
    """Devuelve [fecha, hora] de la celda de fecha."""
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]


def booster_version(table_cells):
    """Versión del booster: cadenas en posiciones pares, sin la última (referencia)."""
    return "".join([bv for i, bv in enumerate(table_cells.strings) if i % 2 == 0][0:-1])


def landing_status(table_cells):
    return [i for i in table_cells.strings][0]


def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    return mass[0 : mass.find("kg") + 2] if mass else 0


def extract_column_from_header(row):
    """Nombre de columna limpio a partir de un <th>."""
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = " ".join(row.contents)
    if not column_name.strip().isdigit():
        return column_name.strip()
    return None


def cell_text(cell):
    """Texto del primer enlace de la celda o, si no hay enlace, su texto plano."""
    if cell.a is not None and cell.a.string:
        return cell.a.string.strip()
    return cell.get_text(" ", strip=True)

## 2. Descarga de la página y creación del objeto BeautifulSoup

In [4]:
try:
    page = requests.get(STATIC_URL, headers=HEADERS, timeout=60)
    page.raise_for_status()
    soup = BeautifulSoup(page.text, "html.parser")
    print("Título de la página:", soup.title.string)
    scraping_ok = True
except requests.RequestException as exc:
    scraping_ok = False
    print("No se pudo descargar la página:", exc)

Título de la página: List of Falcon 9 and Falcon Heavy launches - Wikipedia


## 3. Nombres de columna de la tabla de lanzamientos

In [5]:
if scraping_ok:
    html_tables = soup.find_all("table", "wikitable plainrowheaders collapsible")
    print("Tablas de lanzamientos encontradas:", len(html_tables))
    first_launch_table = html_tables[0]
    column_names = []
    for th in first_launch_table.find_all("th"):
        name = extract_column_from_header(th)
        if name:
            column_names.append(name)
    print(column_names)

Tablas de lanzamientos encontradas: 9
['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## 4. Recorrido de todas las tablas y extracción fila por fila

In [6]:
launch_dict = {key: [] for key in [
    "Flight No.", "Date", "Time", "Version Booster", "Launch site", "Payload",
    "Payload mass", "Orbit", "Customer", "Launch outcome", "Booster landing",
]}

if scraping_ok:
    for table in html_tables:
        for rows in table.find_all("tr"):
            flag = False
            if rows.th and rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
            row = rows.find_all("td")
            if not flag or len(row) < 9:
                continue
            date, time = (date_time(row[0]) + ["", ""])[:2]
            bv = booster_version(row[1]) or cell_text(row[1])
            launch_dict["Flight No."].append(flight_number)
            launch_dict["Date"].append(date.strip(","))
            launch_dict["Time"].append(time)
            launch_dict["Version Booster"].append(re.sub(r"\[\d+\]", "", bv).strip())
            launch_dict["Launch site"].append(cell_text(row[2]))
            launch_dict["Payload"].append(cell_text(row[3]))
            launch_dict["Payload mass"].append(get_mass(row[4]))
            launch_dict["Orbit"].append(cell_text(row[5]))
            launch_dict["Customer"].append(cell_text(row[6]))
            launch_dict["Launch outcome"].append(list(row[7].strings)[0].strip())
            launch_dict["Booster landing"].append(landing_status(row[8]).strip())

    df = pd.DataFrame(launch_dict)
    source = "Wikipedia oldid=1027686922"
else:
    print("Usando el respaldo oficial del curso:", FALLBACK_CSV_URL)
    df = pd.read_csv(FALLBACK_CSV_URL)
    source = "IBM spacex_web_scraped.csv (respaldo)"

print(df.shape)
df.head(10)

(121, 11)


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,4 June 2010,18:45,F9 v1.07B0003.1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure
1,2,8 December 2010,15:43,F9 v1.07B0004.1,CCAFS,Dragon,0,LEO,NASA,Success,Failure
2,3,22 May 2012,07:44,F9 v1.07B0005.1,CCAFS,Dragon,525 kg,LEO,NASA,Success,No
3,4,8 October 2012,00:35,F9 v1.07B0006.1,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,No attempt
4,5,1 March 2013,15:10,F9 v1.07B0007.1,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,No
5,6,29 September 2013,16:00,F9 v1.17B1003,VAFB,CASSIOPE,500 kg,Polar orbit,MDA,Success,Uncontrolled
6,7,3 December 2013,22:41,F9 v1.1,CCAFS,SES-8,"3,170 kg",GTO,SES,Success,No
7,8,6 January 2014,22:06,F9 v1.1,CCAFS,Thaicom 6,"3,325 kg",GTO,Thaicom,Success,No
8,9,18 April 2014,19:25,F9 v1.1,Cape Canaveral,SpaceX CRS-3,"2,296 kg",LEO,NASA,Success,Controlled
9,10,14 July 2014,15:15,F9 v1.1,Cape Canaveral,Orbcomm-OG2,"1,316 kg",LEO,Orbcomm,Success,Controlled


In [7]:
df.to_csv(DATA_DIR / "spacex_web_scraped.csv", index=False)
summary = {"source": source, "rows": int(len(df)), "columns": list(df.columns)}
(RESULTS_DIR / "data_collection_scraping.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
summary

{'source': 'Wikipedia oldid=1027686922',
 'rows': 121,
 'columns': ['Flight No.',
  'Date',
  'Time',
  'Version Booster',
  'Launch site',
  'Payload',
  'Payload mass',
  'Orbit',
  'Customer',
  'Launch outcome',
  'Booster landing']}

## Conclusión

El scraping produce una segunda fuente independiente (fecha, booster, sitio, carga, órbita, cliente y
resultados del lanzamiento y del aterrizaje) que sirve para contrastar y complementar los datos de la API.